# Generators

A generator is a special type of function that produces values **one at a time** instead of producing all values at once.

Unlike a normal function, a generator remembers where it stopped and continues execution from that point the next time it is called.

Generators are memory-efficient and are widely used in:

- Data Engineering
- ETL Pipelines
- File Processing
- Streaming Data
- Large Dataset Processing

## Why Do We Need Generators?

Suppose you need to retrieve a sequence of integers from \(1\) to \(5\). A standard function accomplishes this by pre-computing and returning a collection:

```python
def get_numbers():
    return [1, 2, 3, 4, 5]

numbers = get_numbers()
for num in numbers:
    print(num)
```

This approach works perfectly for microscopic datasets. However, consider what happens when your pipeline scales to require **1,000,000,000 numbers**. 

Returning a traditional list forces Python to allocate system resources for every single element upfront:

```text
[ Create Memory List ] ──► Store 1 Billion Elements ──► Return List ──► Begin Processing
```

This operational strategy creates an enormous memory bottleneck, frequently resulting in out-of-memory crashes on production servers.


## The Stream-Based Alternative

Instead of initializing the entire multi-gigabyte dataset inside system memory at once, a more efficient strategy is to produce and process values one at a time on demand:

```text
Produce Value 1 ──► Process Value 1 ──► Discard From Memory
      │
      ▼
Produce Value 2 ──► Process Value 2 ──► Discard From Memory
      │
      ▼
Produce Value 3 ──► Process Value 3 ──► Discard From Memory
```

Using this architecture, only **one single value** exists in your active workspace at any fractional point in time. This lazy, on-demand generation pattern is exactly how **Generators** optimize software applications.


In [1]:
import sys

# Demonstrating the massive memory discrepancy between Lists and Generators
# 1. A list allocates memory for all elements immediately
list_numbers = [num for num in range(1, 100000)]
list_memory = sys.getsizeof(list_numbers)

# 2. A generator expression stores only the state logic, evaluating lazily
generator_numbers = (num for num in range(1, 100000))
generator_memory = sys.getsizeof(generator_numbers)

print("--- Memory Footprint Evaluation ---")
print(f"List Structure Allocation:      {list_memory:,} bytes")
print(f"Generator Structure Allocation: {generator_memory:,} bytes")
print(f"Generator Efficiency Savings:  {((list_memory - generator_memory) / list_memory) * 100:.2f}%")


--- Memory Footprint Evaluation ---
List Structure Allocation:      800,984 bytes
Generator Structure Allocation: 192 bytes
Generator Efficiency Savings:  99.98%


## Normal Function vs. Generator Function

### Normal Function
A regular function runs from top to bottom and sends back all data at once using `return`. 

```python
def numbers():
    return [1, 2, 3]

result = numbers()
print(result)
```

**Output**:
```text
[1, 2, 3]
```

**Key Behavior**: The function finishes immediately after hitting the `return` line and disappears from memory.


### Generator Function
A generator function uses the **`yield`** keyword instead of `return`.

```python
def numbers():
    yield 1
    yield 2
    yield 3

result = numbers()
print(result)
```

**Output**:
```text
<generator object numbers at 0x...>
```

**Key Behavior**: Notice that no actual numbers are printed! Simply calling the function does not run the code inside it yet. It just creates a **generator object** that is ready to work.


### Getting Values from a Generator
Because **generators are iterators**, you use the built-in `next()` function to ask them for one value at a time.

```python
gen = numbers()

print(next(gen))  # Output: 1
print(next(gen))  # Output: 2
print(next(gen))  # Output: 3
print(next(gen))  # Raises: StopIteration
```

Once there are no more values left to give, the generator raises a `StopIteration` alert to signal that it is finished.


## How Is `yield` Different from `return`?

### 1. The `return` behavior
When a function hits a `return` statement, it exits completely. Any lines written below it are completely ignored.

```python
def test():
    print("A")
    return 10
    print("B")  # This never executes!
```

### 2. The `yield` behavior
When a function hits a `yield` statement, it provides the value and **pauses** right there. It does not exit or delete itself.

```python
def test():
    print("A")
    yield 10

    print("B")
    yield 20

    print("C")
```


### Step-by-Step Execution Tracking

Let's watch how the code runs line by line across multiple `next()` calls:

* **First Call (`next(gen)`)**:
  * Python enters the function, prints `"A"`, and hits `yield 10`. 
  * It returns `10` and pauses on that exact spot.
  * **Output**: `A`, then `10`

* **Second Call (`next(gen)`)**:
  * **Important**: The function does *not* start over from the beginning. It wakes up right where it stopped!
  * It continues down, prints `"B"`, and hits `yield 20`. 
  * It returns `20` and pauses again.
  * **Output**: `B`, then `20`

* **Third Call (`next(gen)`)**:
  * It wakes up, prints `"C"`, runs out of code, and drops out with a `StopIteration` alert.
  * **Output**: `C`, then `StopIteration`


### How a Generator Works Internally

```text
  Start ──► Print "A" ──► yield 10 (Pause) ──► Resume ──► Print "B" ──► yield 20 (Pause) ──► Resume ──► Print "C" ──► Finish
```

A generator acts like a smart machine that explicitly remembers:
1. **The current line** where it paused.
2. **The local variables** inside it.
3. **The exact execution state**.

This unique tracking memory is what makes generators so special.


In [2]:
# Create the generator function matching the example
def test_generator():
    print("[LOG] Starting step A")
    yield 10

    print("[LOG] Waking up and starting step B")
    yield 20

    print("[LOG] Finalizing step C")

# Initialize the generator object instance
gen = test_generator()

print("--- Triggering First Next Call ---")
val1 = next(gen)
print("Returned Value:", val1)

print("\n--- Triggering Second Next Call ---")
val2 = next(gen)
print("Returned Value:", val2)

print("\n--- Triggering Third Next Call ---")
try:
    next(gen)
except StopIteration:
    print("Generator successfully completed and exited via StopIteration.")


--- Triggering First Next Call ---
[LOG] Starting step A
Returned Value: 10

--- Triggering Second Next Call ---
[LOG] Waking up and starting step B
Returned Value: 20

--- Triggering Third Next Call ---
[LOG] Finalizing step C
Generator successfully completed and exited via StopIteration.


## Using a `for` Loop with Generators

In real-world development, you rarely call `next()` manually. Instead, you pass the generator directly into a standard loop.

```python
def numbers():
    yield 1
    yield 2
    yield 3

for num in numbers():
    print(num)
```

**Output**:
```text
1
2
3
```

**Key Behavior**: The `for` loop automatically manages the generator behind the scenes. It systematically calls `next()` for you and stops looping the exact moment it catches a `StopIteration` alert.


## List vs. Generator

| List | Generator |
| :--- | :--- |
| Stores all values in memory at the same time. | Produces only one value at a time on demand. |
| Higher memory (RAM) usage. | Extremely low memory (RAM) usage. |
| Immediate creation of all data elements. | Lazy creation (values are computed only when asked). |
| Faster random access (you can jump to any index instantly). | Strict sequential access (you must read data in order). |


## Large Dataset Example

### The Bad Way (High Memory Burden)
Loading a complete dataset into memory before processing it creates a massive data bottleneck:
```python
# Avoid this for large files
rows = load_entire_csv()
for row in rows:
    process(row)
```
```text
Entire CSV File ──► Loaded into RAM at once ──► Risk of out-of-memory crash
```

### The Good Way (Stream-Based Generator)
Using a generator streams the data line by line, maintaining a flat memory usage profile:
```python
# Do this instead
for row in read_csv_generator():
    process(row)
```
```text
One Row ──► Processed ──► Discarded ──► Next Row Loaded
```
This precise memory-saving optimization is why generators are heavily used in **ETL (Extract, Transform, Load)** pipelines.


## Generator Expressions

Just like you can build a list quickly using **List Comprehension**, you can build a generator instantly using a **Generator Expression**.

The only syntax difference is replacing the square brackets `[]` with regular parentheses `()`:

* **List Comprehension**:
  ```python
  numbers_list = [x*x for x in range(5)]
  print(numbers_list)  # Output: [0, 1, 4, 9, 16]
  ```
* **Generator Expression**:
  ```python
  numbers_gen = (x*x for x in range(5))
  print(numbers_gen)   # Output: <generator object ...>
  ```

### Accessing the Values
To see the results calculated by your generator expression, loop over it exactly like a normal collection:
```python
for value in numbers_gen:
    print(value)
```


In [3]:
numbers_list = [x*x for x in range(5)]
print(numbers_list) 

numbers_gen = (x*x for x in range(5))
print(numbers_gen)

[0, 1, 4, 9, 16]
<generator object <genexpr> at 0x000002561D92F780>


In [4]:
for value in numbers_gen:
    print(value)

0
1
4
9
16


## When Should You Use Generators?

### Use generators when:
* Reading exceptionally large source files or logs.
* Streaming active API responses over web networks.
* Processing millions of data rows or records sequentially.
* Building data transformation and ETL software pipelines.

### Avoid generators when:
* You need random indexing (running `gen[5]` is not possible and will throw an error).
* You need to loop through the exact same data multiple times without resetting or recreating the generator.
* Your dataset is very small and basic simplicity is your primary goal.


In [5]:
import sys

# 1. Initialize a live generator expression
squares_generator = (x * x for x in range(5))

print("--- Inspecting the Generator Expression ---")
print("Object Blueprint:", squares_generator)

print("\n--- Iterating Using a for Loop ---")
for square in squares_generator:
    print("Streamed Element Value:", square)

# 2. Demonstrating that generators run dry after a single pass
print("\n--- Attempting a Second Pass ---")
count = 0
for square in squares_generator:
    count += 1
print(f"Total elements read on second pass: {count} (Generators cannot be reused!)")


--- Inspecting the Generator Expression ---
Object Blueprint: <generator object <genexpr> at 0x000002561DCBC6C0>

--- Iterating Using a for Loop ---
Streamed Element Value: 0
Streamed Element Value: 1
Streamed Element Value: 4
Streamed Element Value: 9
Streamed Element Value: 16

--- Attempting a Second Pass ---
Total elements read on second pass: 0 (Generators cannot be reused!)
